# 06: Volt-VAr-induced curtailment: Methods A, B and C

Three methods that **bracket** the same quantity:

| | | |
|---|---|---|
| **A** | apparent-limit symptom scan | **upper bound**. Assumes every symptom interval was sun-limited |
| **B** | counterfactual attribution | **lower bound**. Counts only what clear-sky GHI confirms |
| **C** | derating-flag corroboration | not an estimate. Independent *label* to test A and B against |

A and B are reported as a **range** and are not reconciled. (note that averaging them wouldinvent a number neither method supports).

## sign caveat:
the reactive sign is not fully resolved

`exclude_polarity_suspect=True` (the default) drops the sites `se_adverse` flags as
`polarity_suspect` before scanning. That is a **cohort restriction**, not a correction —
it removes sites whose direction cannot be trusted rather than silently flipping them.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents)
                  if (p / "oem_analysis").is_dir() and (p / "bms_sa_review").is_dir()), None)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from oem_analysis.config import se_config as C
from oem_analysis.lib import se_store, se_contract as contract, se_params
pd.set_option("display.max_columns", None); pd.set_option("display.width", 220)
con = se_store.connect()
config, params = se_params.CONFIG, se_params.PARAMS
from oem_analysis.lib import se_curtailment as cu
from oem_analysis.lib import se_plots as plots
from oem_analysis.lib import se_adverse as adv

adverse = adv.classify_adverse_sites(con, config)
display(adverse.adverse_class.value_counts())
display(contract.manifest(config, params).query("section in ('detection','basis')"))

## Method A: apparent-limit symptom scan

An interval is flagged when the inverter is **absorbing** reactive power **and** sitting on its apparent-power circle:

```
Q_kvar < 0  AND  sqrt(P² + Q²) >= s_limit − tol × capacity
```

The proxy energy is the **headroom displacement**, `s_limit − sqrt(s_limit² − Q²)`: The kW of circle room that reactive absorption consumed.

**Two biases, both upward.** 

- It assumes the inverter would have used that headroom, which is only true when the sun was available. 
- And `s_limit` is `s_99`, an *observed* p99: a site that never approached its true inverter limit gets a low `s_limit`, so the test fires more readily. 

Method A here is an even looser upper bound givan it has no nameplate to anchor to.

In [ ]:
method_a = cu.method_a_site_year(con, config, params, adverse=adverse, exclude_polarity_suspect=False)
display(cu.method_a_summary(method_a, config).T)

context = cu.eligible_context(con, config, params, adverse=adverse)
print(f"Eligible: {int(context.n_eligible_intervals.sum()):,} intervals "
      f"across {context.site_alias.nunique():,} sites")
print(f"Measured energy in those intervals: {context.measured_kWh.sum():,.0f} kWh")

##  Method B: counterfactual attribution

**Requires the GHI counterfactual.** The cell below raises a clear error if `se_uncurtailedpv` is absent rather than returning zeros.

Four evidence tiers, each strictly narrower than the last:

```
1  absorbing Q                                Q < 0
2  + apparent-limit symptom                   on the S-circle
3  + counterfactual above measured-Q headroom uncurtailed_P > pmax
4  + attributable displacement > 0            the reportable number
```

In [ ]:
try:
    method_b = cu.method_b_site_year(con, config, params, adverse=adverse)
    display(cu.method_b_summary(method_b, config).T)
    display(cu.evidence_tiers(method_b))
except FileNotFoundError as exc:
    method_b = None
    print("Method B not runnable yet:\n"); print(exc)

## Fleet-level reporting

### Two denominators, three decimal places apart

`5.374 MWh` is the numerator throughout. What it is divided by changes the answer by
orders of magnitude, so a bare percentage means nothing without naming the denominator:

| Denominator | What it answers | Rough scale |
|---|---|---|
| **Eligible** potential generation — `sum(uncurtailed_P)` over covered 240–253 V peak intervals | *Of the generation that was at risk, how much was lost?* | ~5,600 MWh |
| **All** fleet generation — every interval, every site | *Of everything the fleet made, what did Volt-VAr cost?* | far larger |

The first is the exposure-conditional rate and is the fairer measure of how the mechanism
behaves when it acts. The second is the fleet-assurance number and is what a reader
assumes when handed an unqualified "% of generation".

**For the "all" denominator, prefer `measured_MWh`.** It is metered energy and needs no
model, so it cannot be wrong. `potential_MWh` coalesces to measured wherever no
counterfactual exists — and fleet-wide coverage is roughly 12%, versus ~94% inside the
eligible band, because the GHI model only fits where clear-sky training days exist. On the
uncovered majority the fallback is a *lower* bound on potential, so `potential_MWh` is
understated and any percentage using it is overstated. `pct_intervals_covered` is printed
so that bias is visible rather than assumed away.

### Volt-VAr fleet report

- **cohort sites (outer denominator)**: every site in the residential-scale
  cohort that passes the basic study filters (months, capacity bounds, minimum
  days observed, night-generation-anomaly exclusion). No voltage condition at
  all.
- **sites with any eligible interval**: sites with at least one interval in
  the *curtailment-eligible* population: `240 V < V < 253 V` (open interval,
  mean-of-phases) **and** peak-solar hours **and** valid P/Q/V readings **and**
  `s_99 > 0`. This is **not** the same population as Volt-VAr *conformance*
  scoring (which additionally requires `|P| ≥ 0.2×S`).
- **sites showing the V-VAr symptom (Tier 2)**: sites with at least one
  interval where `Q_kvar < 0` (absorbing reactive power) **and**
  `sqrt(P² + Q²) ≥ s_limit − tolerance` (sitting on/near the apparent-power
  circle). This is a *symptom*, not confirmed curtailment: it flags "this
  interval looks like the inverter may have been constrained by its apparent-
  power limit", which Method A treats as its whole basis and Method B treats
  as one necessary (but not sufficient) condition.
- **sites with effective energy loss (Tier 4)**: sites with at least one
  interval where `attributed_kW > 0`, i.e. the counterfactual-confirmed,
  quantified shortfall: symptom present **and** the modelled uncurtailed
  generation (`uncurtailed_P`) exceeds what the measured Q's apparent-power
  headroom (`pmax_measured_q_kw`) would allow. This is the population the
  headline kWh number is summed over.
- **eligible timestamps (240-253 V, peak)**: count of intervals in the
  eligible population above. (The "clear-sky" qualifier in the on-screen label
  is a bit ahead of the code — eligibility itself has no clear-sky filter; that
  restriction is what determines the *next* row, `counterfactual coverage`.)
- **counterfactual-covered timestamps**: of the eligible timestamps, how many
  also fell on a site/time-of-day bin the GHI model could fit (roughly: how
  many were on a clear-sky-classifiable day). Curtailment can only be assessed
  where this is true.
- **symptom timestamps**: Tier 2 interval count: `Q < 0` and near the
  apparent-power circle, fleet-wide.
- **curtailed timestamps (attributed > 0)**: Tier 4 interval count: the
  subset of symptom intervals the counterfactual actually confirms as lost
  energy.
- **estimated V-VAr curtailment (kWh / MWh)**: `sum(attributed_kW) ×
  interval_h`, i.e. the quantified Tier-4 shortfall, fleet-wide. This is
  Method B's number. A lower bound, since it counts only counterfactual-
  confirmed loss.
- **eligible modelled PV generation (MWh)**: `sum(uncurtailed_P)` over the
  *covered* eligible intervals only (the 240-253V/peak-hours/counterfactual-
  available population). This is the "at risk" denominator.
- **curtailment as % of ELIGIBLE potential generation**: curtailment ÷ the
  row above. Answers "of the generation that was at risk (in the 240-253 V
  band), how much was lost?". A narrow, exposure-conditional rate.
- **ALL fleet measured / potential generation (MWh)**: the second, much wider
  denominator: every interval, every cohort site, no voltage restriction at
  all. "Measured" is pure metered energy (`sum(P_kW)`, cannot be wrong).
  "Potential" coalesces to the counterfactual where available and to measured
  elsewhere, so it's a lower bound wherever coverage is thin fleet-wide (~74%
  here, versus ~94% inside the eligible band).
- **curtailment as % of ALL measured / potential generation**: curtailment ÷
  one of the two rows above. Answers "of everything the fleet made, what did
  Volt-VAr cost?" — the fleet-assurance number. Prefer the "measured" version:
  it needs no model and can't be wrong; the "potential" version is understated
  wherever coverage is thin.

In [ ]:
cohort_sites = cu.cohort_site_count(con, config)
print(f"residential-scale cohort: {cohort_sites:,} sites")

# The SECOND denominator: all cohort generation, not just the screened 240-253 V band.
# This scans every interval in the store, so it is the slowest cell in the notebook.
fleet_generation = cu.fleet_potential_generation(con, config)
display(fleet_generation)

if method_b is not None:
    fleet_report = cu.fleet_curtailment_report(
        method_b, config, cohort_sites, fleet_generation)
    display(fleet_report)
else:
    fleet_report = None
    print("\nNeeds Method B -- build the D12 counterfactual first (notebook 04, section 5).")

### Concentration of loss

In [ ]:
if method_b is not None:
    concentration = cu.curtailment_concentration(method_b, config)
    display(concentration)
    print(f"affected sites: {concentration.attrs['n_affected_sites']:,}   "
          f"total: {concentration.attrs['total_kWh']:,.1f} kWh")
else:
    concentration = None

In [ ]:
# Figure 37: the Lorenz curve behind the table above.
# Sites are ranked worst-first so the curve reads as "the top X% account for Y%".
# That is the reverse of a textbook Lorenz curve, so it bows ABOVE the diagonal.
if concentration is not None:
    display(plots.plot_curtailment_lorenz(
        concentration,
        "Volt-VAr: concentration of estimated curtailment",
        f"{concentration.attrs['n_affected_sites']:,} affected sites"))

In [ ]:
# The A-B range, which the paragraph above deliberately does not collapse into one number.
if method_b is not None:
    a_kwh = method_a.headroom_displacement_kw_sum.sum() * config.interval_h
    b_kwh = method_b.attributed_kw_sum.sum() * config.interval_h
    print(f"Method A (upper bound, assumes every symptom interval was sun-limited): "
          f"{a_kwh / 1000:8.3f} MWh")
    print(f"Method B (lower bound, counterfactual-confirmed only):                  "
          f"{b_kwh / 1000:8.3f} MWh")
    print(f"ratio A/B: {a_kwh / b_kwh:.1f}x" if b_kwh else "")

In [ ]:
if fleet_report is not None:
    fleet_report.to_csv(C.ARTEFACT_DIR / "fleet_curtailment_report.csv", index=False)
    concentration.to_csv(C.ARTEFACT_DIR / "curtailment_concentration.csv", index=False)
    (C.ARTEFACT_DIR / "fleet_curtailment_narrative.txt").write_text(narrative)
    print(f"-> {C.ARTEFACT_DIR / 'fleet_curtailment_report.csv'}")
    print(f"-> {C.ARTEFACT_DIR / 'curtailment_concentration.csv'}")
    print(f"-> {C.ARTEFACT_DIR / 'fleet_curtailment_narrative.txt'}")

## Method C: derating-flag corroboration

SolarEdge reports `derating_active` per interval. 

This is **not a third estimate**, it is an independent label for the thing Method A is trying to infer.

**Read precision, not recall.** The raw flag is `1.0` or NULL, never `0.0`, so "not
derating" and "not reported" are indistinguishable. *Of the intervals the inverter says
it was derating, what fraction did Method A catch?* is sound. *Of the intervals Method A
missed, how many were really derating?* is unanswerable.

The flag is also not Volt-VAr specific — thermal limits, DC clipping and export control
all set it. Agreement corroborates "something limited output", not the mechanism.

In [ ]:
confusion = cu.method_c_confusion(con, config, params, adverse=adverse)
display(confusion)
counts = confusion.attrs["counts"]
print(f"Method A symptom AND derating flag : {counts['tp']:,}")
print(f"Method A symptom, no flag          : {counts['fp']:,}")
print(f"Flag but no Method A symptom       : {counts['fn']:,}")
print(f"\nPrecision (interpretable)          : {confusion.attrs['precision']:.4f}")
print("Recall is NOT interpretable — see above.")

### Is the flag tracking Volt-VAr or Volt-Watt?

The discriminator. If the derating rate only climbs above **253 V** it is tracking
Volt-Watt and says nothing about the 240–253 V band where Method A operates. A rise
*within* 240–253 V is what would make it corroborate a Volt-VAr claim.

In [ ]:
by_v = cu.method_c_by_voltage(con, config)
display(by_v)

band = by_v[(by_v.v_bin >= 240) & (by_v.v_bin < 253)]
print(f"Derating rate at 240 V: {band.pct_derating.iloc[0]:.2f}%")
print(f"Derating rate at 252 V: {band.pct_derating.iloc[-1]:.2f}%")
print("A rise within this band supports a Volt-VAr reading; a flat line does not.")

## Volt-Watt curtailment

For every interval with **V > 253 V** (Volt-Watt active), with
`ceiling = vw_max_p(V, s_99) + 4%`:

```
is_curtailed        =  uncurtailed_P > ceiling   AND   P_kW < ceiling

curtailed_kW        =  uncurtailed_P − P_kW          # total energy foregone
  mandated_kW       =  uncurtailed_P − ceiling       #   required by the standard
  over_reduction_kW =  ceiling − P_kW                #   shed beyond the requirement
```

`curtailed_kW = mandated_kW + over_reduction_kW` by construction.

The counterfactual is floored at measured P (`uncurtailed_P = greatest(prediction, P_kW)`),
so `curtailed_kW ≥ 0` always and the estimate can never claim a site produced less than
was observed.

### Lower bound due to GHI coverage

Intervals with no `uncurtailed_P` contribute **zero**, not "unknown". Every gap in the counterfactual pushes the total down.

In [ ]:
try:
    vw_curt = cu.voltwatt_curtailment_site_year(con, config, params)
    vw_curt_summary = cu.voltwatt_curtailment_summary(vw_curt, config)
    display(vw_curt_summary.T)
    cu.voltwatt_curtailment_note(vw_curt_summary)

    vw_curt.to_csv(C.ARTEFACT_DIR / "voltwatt_curtailment_by_site.csv", index=False)
    print(f"\nPer-site -> {C.ARTEFACT_DIR / 'voltwatt_curtailment_by_site.csv'}")
except FileNotFoundError as exc:
    vw_curt = None
    print("Volt-Watt curtailment not runnable yet:\n"); print(exc)

#### Which sites, and how much

`response_opportunity_count` is the honest denominator for "did this site curtail":
intervals where the counterfactual says power was available above the ceiling. A site with
zero response opportunities was never asked, and its zero curtailment says nothing.

In [ ]:
if vw_curt is not None and len(vw_curt):
    top = vw_curt[vw_curt.curtailed_count > 0].copy()
    top["curtailed_kWh"] = (top.curtailed_kw_sum * C.INTERVAL_H).round(2)
    top["over_reduction_kWh"] = (top.over_reduction_kw_sum * C.INTERVAL_H).round(2)
    top["pct_opportunities_curtailed"] = (
        100 * top.curtailed_count / top.response_opportunity_count.replace(0, pd.NA)
    ).round(1)
    display(top.nlargest(15, "curtailed_kWh")[
        ["site_alias", "state", "rating_kva", "exposed_count",
         "counterfactual_covered_count", "response_opportunity_count",
         "curtailed_count", "pct_opportunities_curtailed",
         "curtailed_kWh", "over_reduction_kWh"]])

    print(f"\n{len(top):,} of {len(vw_curt):,} exposed sites show any Volt-Watt "
          f"curtailment.")
    print(f"{int((vw_curt.response_opportunity_count == 0).sum()):,} sites had NO "
          f"response opportunity at all — their zero is uninformative.")

#### Fleet-level reporting: Volt-Watt

The same reporting frame as the Volt-VAr section, with three deliberate differences in
what the rows mean:

- **"exposed", not "eligible".** Volt-VAr is assessed over 240–253 V, Volt-Watt only above
  253 V. The two populations are disjoint by construction, so the totals are additive
  rather than overlapping — that separation is what makes either attributable.
- **No symptom tier.** Volt-Watt needs no proxy. The standard states the ceiling, so
  "the sun could have delivered more than the ceiling allowed" is directly checkable.
  `response_opportunity` replaces the symptom count.
- **The total splits.** `mandated` is the designed cost of the standard; `over_reduction`
  is what inverters shed beyond what was asked. Only the second is a compliance finding —
  reporting the sum alone conflates a working standard with a badly tuned inverter.

### Volt-Watt fleet report

- **cohort sites (outer denominator)**: same population as the Volt-VAr
  report's row 0: every site passing the basic cohort filters, no voltage
  condition.
- **sites exposed to Volt-Watt (V > 253 V)**: sites with at least one
  interval above 253 V (mean-of-phases). Unlike Volt-VAr's eligible population,
  this has **no peak-hours restriction** — Volt-Watt is checked whenever V is
  high, any time of day.
- **sites with a response opportunity**: sites with at least one exposed
  interval where the modelled `uncurtailed_P` exceeds the mandated ceiling
  (`vw_max_p(V, S) + 4%` tolerance): i.e. there was enough sun that the
  standard *should* have required a reduction. This does **not** check whether
  the site actually reduced output.
- **sites with effective energy loss**: the stricter condition: at least one
  interval where `uncurtailed_P` exceeded the ceiling **and** measured `P_kW`
  was actually below it — i.e. curtailment was observably happening, not just
  possible. (Sites in "response opportunity" but not here had the opportunity
  but never showed a measured pull-back in any of those intervals — worth
  cross-checking against the formal Volt-Watt conformance verdict in
  `se_conformance.py` before calling this "non-conformant": it's a different
  statistic, computed a different way.)
- **exposed / counterfactual-covered / response-opportunity / curtailed
  timestamps** — the same four population narrowings as above, at the
  interval level rather than the site level.
- **estimated Volt-Watt curtailment (kWh / MWh)**: `sum(curtailed_kW) ×
  interval_h`, where `curtailed_kW = uncurtailed_P − P_kW`, counted only on
  curtailed intervals. By construction this splits into:
  - **mandated (MWh)** — `uncurtailed_P − ceiling`: the part the standard
    actually required to be shed.
  - **over-reduction (MWh)** — `ceiling − P_kW`: the part shed *beyond* the
    requirement. A large share here points at inverter tuning, not compliance.
- **exposed potential PV generation (MWh)**: `sum(coalesce(uncurtailed_P,
  P_kW))` over every exposed interval (counterfactual where available,
  measured as a fallback elsewhere) — the "at risk" denominator for Volt-Watt.
- **curtailment as % of exposed potential**: curtailment ÷ the row above.
- **ALL fleet measured / potential generation (MWh)** and **curtailment as %
  of ALL measured / potential generation**: identical definitions and the
  identical underlying numbers as the Volt-VAr report (same
  `fleet_potential_generation()` call, run once and shared between both
  reports) — only the curtailment numerator changes. Lets you read "what did
  Volt-Watt cost the whole fleet" side by side with the Volt-VAr figure above.

In [ ]:
if vw_curt is not None and len(vw_curt):
    vw_fleet_report = cu.voltwatt_fleet_report(vw_curt, config, cohort_sites, fleet_generation)
    display(vw_fleet_report)
else:
    vw_fleet_report = None

In [ ]:
if vw_curt is not None and len(vw_curt):
    vw_concentration = cu.curtailment_concentration(vw_curt, config, mode="voltwatt")
    display(vw_concentration)
    print(f"affected sites: {vw_concentration.attrs['n_affected_sites']:,}   "
          f"total: {vw_concentration.attrs['total_kWh']:,.1f} kWh")
    display(plots.plot_curtailment_lorenz(
        vw_concentration,
        "Volt-Watt: concentration of estimated curtailment",
        f"{vw_concentration.attrs['n_affected_sites']:,} affected sites"))
else:
    vw_concentration = None

#### Volt-VAr vs Volt-Watt curtailment

Two different mechanisms competing for the same headroom. Volt-VAr curtailment is a side
effect of reactive absorption; Volt-Watt curtailment is the standard doing exactly what it
says. Comparing the energies says which one actually costs generation on this fleet —
subject, as always, to Method B's coverage and this calculation's coverage being
different numbers.

In [ ]:
if vw_curt is not None and method_b is not None:
    vv_kwh = float(method_b.attributed_kw_sum.sum()) * C.INTERVAL_H
    vw_kwh = float(vw_curt.curtailed_kw_sum.sum()) * C.INTERVAL_H
    display(pd.DataFrame([
        {"mechanism": "Volt-VAr (Method B, counterfactual)",
         "kWh": round(vv_kwh, 2),
         "coverage_note": "attributed only where uncurtailed_P exists"},
        {"mechanism": "Volt-Watt (ceiling vs counterfactual)",
         "kWh": round(vw_kwh, 2),
         "coverage_note": f"{vw_curt_summary.counterfactual_covered.sum():,.0f} of "
                          f"{vw_curt_summary.exposed_intervals.sum():,.0f} exposed "
                          f"intervals covered"},
    ]))
    print("Both are LOWER bounds and their coverage differs — do not read the ratio")
    print("as a physical result until counterfactual coverage is comparable.")
else:
    print("Needs both method_b and vw_curt.")

## A vs B vs C

The range, stated as a range.

In [ ]:
display(cu.method_comparison(method_a, method_b, confusion, config))

method_a.to_csv(C.ARTEFACT_DIR / "method_a_by_site.csv", index=False)
print(f"-> {C.ARTEFACT_DIR / 'method_a_by_site.csv'}")